In [12]:
base = "/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead"
# "/data/big_rim/rsync_dcc_sum/25juneon/2025_09_19/0ctrl_10base10opto_13_54_cam1dead"

In [ ]:
#below works but very length not necessary, so abandon. 
# # %% Detect missing cameras (by checking videos/Camera{i}/metadata.csv)
# import os

# def detect_missing_cameras(base_path: str, n_cams: int = 6):
#     """
#     Return which cameras are present/missing by checking:
#       <base_path>/videos/Camera{i}/metadata.csv  for i=1..n_cams
#     """
#     vid_dir = os.path.join(base_path, "videos")
#     if not os.path.isdir(vid_dir):
#         raise NotADirectoryError(f"videos/ not found under base_path: {vid_dir}")

#     ids = list(range(1, n_cams + 1))
#     paths = [os.path.join(vid_dir, f"Camera{i}", "metadata.csv") for i in ids]
#     exists = [os.path.isfile(p) for p in paths]

#     present = [i for i, e in zip(ids, exists) if e]
#     missing = [i for i, e in zip(ids, exists) if not e]

#     summary = {
#         "present": present,
#         "missing": missing,
#         "n_present": len(present),
#         "n_missing": len(missing),
#         "checked_paths": {i: p for i, p in zip(ids, paths)},
#     }

#     # Simple printout for quick testing
#     print(f"Present ({summary['n_present']}/{n_cams}): {present}")
#     print(f"Missing ({summary['n_missing']}): {missing}")

#     return summary

# # Example:
# base_path = base
# s = detect_missing_cameras(base_path)

# s

Present (5/6): [2, 3, 4, 5, 6]
Missing (1): [1]


{'present': [2, 3, 4, 5, 6],
 'missing': [1],
 'n_present': 5,
 'n_missing': 1,
 'checked_paths': {1: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera1/metadata.csv',
  2: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera2/metadata.csv',
  3: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera3/metadata.csv',
  4: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera4/metadata.csv',
  5: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera5/metadata.csv',
  6: '/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/videos/Camera6/metadata.csv'}}

In [19]:
# %% Missing-cam utilities (modular, Python <3.10 friendly)
import os, glob
from typing import Optional, List, Tuple
import numpy as np
import scipy.io as sio


# ---------- 1) DETECT ----------
def detect_missing_cameras(base_path: str, n_cams: int = 6) -> Tuple[List[int], List[int]]:
    vid_dir = os.path.join(base_path, "videos")
    ids = list(range(1, n_cams + 1))
    present = [i for i in ids if os.path.isfile(os.path.join(vid_dir, f"Camera{i}", "metadata.csv"))]
    missing = [i for i in ids if i not in present]
    return present, missing


# ---------- small helpers ----------
def _find_label3d_mat(base_path: str) -> str:
    mats = [p for p in glob.glob(os.path.join(base_path, "*.mat"))
            if os.path.basename(p).lower().endswith("label3d_dannce.mat")]
    if len(mats) != 1:
        raise RuntimeError(f"Expected exactly 1 '*label3d_dannce.mat' in {base_path}, found {len(mats)}: {mats}")
    return mats[0]

def _is_1x1_obj(x) -> bool:
    return isinstance(x, np.ndarray) and x.dtype == object and x.shape == (1, 1)

def _strip1(x):
    return x[0, 0] if _is_1x1_obj(x) else x

def _get_rec(P, i):
    if isinstance(P, np.ndarray) and P.ndim == 2 and P.shape[1] == 1:
        return P[i, 0]
    return P[i]

def _set_if_has(dst, src, field: str):
    if hasattr(dst, field) and hasattr(src, field):
        setattr(dst, field, _strip1(getattr(src, field)))

def _camname_map(m) -> Tuple[List[str], dict]:
    v = np.squeeze(m["camnames"])
    camnames = [str(np.squeeze(x)) for x in (v.tolist() if isinstance(v.tolist(), list) else [v.tolist()])]
    name_to_idx = {nm: i for i, nm in enumerate(camnames)}
    return camnames, name_to_idx

def _camid_to_idx(name_to_idx: dict, cam_id: int) -> int:
    nm = f"Camera{cam_id}"
    if nm not in name_to_idx:
        raise KeyError(f"{nm} not in camnames.")
    return name_to_idx[nm]

# --- add these small helpers ---
def _is_struct_1x1(x) -> bool:
    return isinstance(x, np.ndarray) and (x.dtype.names is not None) and (x.shape == (1, 1))

def _strip1_any(x):
    # unwrap one layer if it's a 1x1 object OR a 1x1 structured array
    if _is_1x1_obj(x) or _is_struct_1x1(x):
        return x[0, 0]
    return x

def _unwrap_sync_one_layer_inplace(sync_arr):
    """Unwrap exactly one layer for each sync cell, in place."""
    if not isinstance(sync_arr, np.ndarray):
        return
    for idx in np.ndindex(sync_arr.shape):
        cell = sync_arr[idx]
        cell2 = _strip1_any(cell)
        if cell2 is not cell:
            sync_arr[idx] = cell2


# ---------- 2) CLONE (no sync edits; final unwrap pass) ----------
def clone_missing_cameras_in_mat(
    base_path: str,
    donor_cam_id: Optional[int] = None,
    n_cams: int = 6,
    fields_to_copy: Tuple[str, ...] = ("K", "RDistort", "TDistort", "r", "t"),
    out_prefix: str = "clonedMissing_",
    precomputed_missing: Optional[List[int]] = None,
    fix_sync_one_layer: bool = True,           # <— new flag
) -> str:
    present, missing = detect_missing_cameras(base_path, n_cams) if precomputed_missing is None \
                       else ([i for i in range(1, n_cams+1) if i not in precomputed_missing], precomputed_missing)

    if not present:
        raise RuntimeError("No live cameras detected.")
    if not missing:
        return _find_label3d_mat(base_path)

    donor = donor_cam_id if (donor_cam_id in present) else present[0]

    in_mat = _find_label3d_mat(base_path)
    m = sio.loadmat(in_mat, struct_as_record=False, squeeze_me=False)
    if "params" not in m or "camnames" not in m:
        raise KeyError("MAT missing 'params' and/or 'camnames'.")

    P = m["params"]
    sync = m.get("sync", None)
    _, name_to_idx = _camname_map(m)

    donor_idx = _camid_to_idx(name_to_idx, donor)
    rec_src = _strip1(_get_rec(P, donor_idx))

    # ---- clone PARAMS only ----
    for cam in missing:
        dst_idx = _camid_to_idx(name_to_idx, cam)
        rec_dst = _strip1(_get_rec(P, dst_idx))

        if isinstance(P, np.ndarray):
            if P.ndim == 2 and P.shape[1] == 1 and _is_1x1_obj(P[dst_idx, 0]):
                P[dst_idx, 0] = rec_dst
            elif P.ndim == 1 and _is_1x1_obj(P[dst_idx]):
                P[dst_idx] = rec_dst

        for f in fields_to_copy:
            _set_if_has(rec_dst, rec_src, f)

    # ---- final unwrap pass for params ----
    if isinstance(P, np.ndarray):
        for idx in np.ndindex(P.shape):
            elem = P[idx]
            if _is_1x1_obj(elem):
                P[idx] = elem[0, 0]
                elem = P[idx]
            for f in fields_to_copy:
                if hasattr(elem, f):
                    setattr(elem, f, _strip1(getattr(elem, f)))

    # ---- normalize sync: unwrap exactly one layer per cell; do NOT change contents ----
    if fix_sync_one_layer and ("sync" in m) and isinstance(sync, np.ndarray):
        _unwrap_sync_one_layer_inplace(sync)

    out_mat = os.path.join(base_path, out_prefix + os.path.basename(in_mat))
    sio.savemat(out_mat, m, do_compression=True, long_field_names=True)
    return out_mat


# ---------------- Example (minimal) ----------------
# base = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_07/2social_mini_20241015pmcr2_single_AO_13_24"
present, missing = detect_missing_cameras(base)
outp = clone_missing_cameras_in_mat(base, precomputed_missing=missing)  # optionally donor_cam_id=2
outp


'/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead/clonedMissing_2025_09_19_0ctrl_10base10opto_13_54_cam1dead_calib_before_label3d_dannce.mat'

In [16]:
# %% Copy/overwrite videos from a donor camera into missing camera folders
import os, shutil
from typing import Optional, List, Tuple

def copy_videos_from_donor(
    base_path: str,
    donor_cam_id: Optional[int] = None,           # if None, picks first present cam
    targets: Optional[List[int]] = None,          # if None, fills missing cams
    n_cams: int = 6,
    video_exts: Tuple[str, ...] = (".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mjpeg", ".mjpg"),
    clean_target: bool = False,
    sidecars: Tuple[str, ...] = tuple(),          # e.g., ("metadata.csv","timestamps.csv")
) -> dict:
    """
    Copy (overwrite) donor camera's video files into target camera folders under:
        <base_path>/videos/Camera{i}
    Returns a summary dict {target_cam_id: {"copied": int, "skipped": int}}.
    """
    # 1) figure out present/missing and targets
    try:
        present, missing = detect_missing_cameras(base_path, n_cams)
    except NameError:
        # fallback without relying on the earlier function
        vids_dir = os.path.join(base_path, "videos")
        ids = list(range(1, n_cams + 1))
        present = [i for i in ids if os.path.isfile(os.path.join(vids_dir, f"Camera{i}", "metadata.csv"))]
        missing = [i for i in ids if i not in present]

    if donor_cam_id is None:
        if not present:
            raise RuntimeError("No live cameras to use as donor.")
        donor_cam_id = present[0]

    if targets is None:
        targets = list(missing)

    # nothing to do
    if not targets:
        return {}

    vids_dir = os.path.join(base_path, "videos")
    donor_dir = os.path.join(vids_dir, f"Camera{donor_cam_id}")
    if not os.path.isdir(donor_dir):
        raise NotADirectoryError(f"Donor folder not found: {donor_dir}")

    # 2) collect donor files (non-recursive)
    def _list_videos(folder: str):
        try:
            names = os.listdir(folder)
        except FileNotFoundError:
            return []
        vids = [n for n in names if n.lower().endswith(video_exts)]
        return vids

    donor_videos = _list_videos(donor_dir)
    if not donor_videos:
        raise FileNotFoundError(f"No donor videos with extensions {video_exts} in {donor_dir}")

    # include sidecars if requested
    donor_sidecars = [s for s in sidecars if os.path.isfile(os.path.join(donor_dir, s))]

    # 3) copy into each target
    summary = {}
    for cam in targets:
        if cam == donor_cam_id:
            continue
        tgt_dir = os.path.join(vids_dir, f"Camera{cam}")
        os.makedirs(tgt_dir, exist_ok=True)

        # optional clean
        if clean_target:
            for n in list(os.listdir(tgt_dir)):
                if n.lower().endswith(video_exts):
                    try:
                        os.remove(os.path.join(tgt_dir, n))
                    except OSError:
                        pass  # ignore failures silently

        copied = 0
        skipped = 0

        # copy videos (overwrite)
        for name in donor_videos:
            src = os.path.join(donor_dir, name)
            dst = os.path.join(tgt_dir, name)
            try:
                shutil.copy2(src, dst)  # overwrite
                copied += 1
            except Exception:
                skipped += 1

        # copy sidecars (optional)
        for s in donor_sidecars:
            src = os.path.join(donor_dir, s)
            dst = os.path.join(tgt_dir, s)
            try:
                shutil.copy2(src, dst)
            except Exception:
                pass

        summary[cam] = {"copied": copied, "skipped": skipped}

    return summary

# Example:
# base = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_07/2social_mini_20241015pmcr2_single_AO_13_24"
# present, missing = detect_missing_cameras(base)
copy_videos_from_donor(base, donor_cam_id=present[0], targets=missing, clean_target=True)


{1: {'copied': 1, 'skipped': 0}}

In [14]:
present, missing

([2, 3, 4, 5, 6], [1])

In [8]:
base

'/data/big_rim/mir_data/5cam_testr/0ctrl_10base10opto_13_54_cam1dead'

In [ ]:
# %% One-call wrapper: detect -> copy videos -> clone params -> (optional) verify
import os, glob
from typing import Optional, List, Dict, Tuple
import numpy as np
import scipy.io as sio

# assumes these already exist in your session:
# - detect_missing_cameras(base_path, n_cams=6) -> (present, missing)
# - copy_videos_from_donor(base_path, donor_cam_id=None, targets=None, n_cams=6, clean_target=True, ...)
# - clone_missing_cameras_in_mat(base_path, donor_cam_id=None, n_cams=6, out_prefix="clonedMissing_", precomputed_missing=None, ...)

def _find_label3d_mat(base_path: str) -> str:
    mats = [p for p in glob.glob(os.path.join(base_path, "*.mat"))
            if os.path.basename(p).lower().endswith("label3d_dannce.mat")]
    if len(mats) != 1:
        raise RuntimeError(f"Expected exactly 1 '*label3d_dannce.mat', found {len(mats)}: {mats}")
    return mats[0]

def _is_1x1_obj(x) -> bool:
    return isinstance(x, np.ndarray) and x.dtype == object and x.shape == (1, 1)

def _strip1(x):
    return x[0, 0] if _is_1x1_obj(x) else x

def _get_rec(P, i):
    if isinstance(P, np.ndarray) and P.ndim == 2 and P.shape[1] == 1:
        return P[i, 0]
    return P[i]

def _camname_map(m) -> Tuple[List[str], dict]:
    v = np.squeeze(m["camnames"])
    camnames = [str(np.squeeze(x)) for x in (v.tolist() if isinstance(v.tolist(), list) else [v.tolist()])]
    name_to_idx = {nm: i for i, nm in enumerate(camnames)}
    return camnames, name_to_idx

def _camid_to_idx(name_to_idx: dict, cam_id: int) -> int:
    nm = f"Camera{cam_id}"
    if nm not in name_to_idx:
        raise KeyError(f"{nm} not in camnames.")
    return name_to_idx[nm]

def _get_sync_cell(sync_arr, idx: int):
    return sync_arr[idx, 0] if (isinstance(sync_arr, np.ndarray) and sync_arr.ndim == 2 and sync_arr.shape[1] == 1) else sync_arr[idx]

def _list_videos(folder: str, video_exts: Tuple[str, ...]) -> List[str]:
    try:
        names = os.listdir(folder)
    except FileNotFoundError:
        return []
    return sorted([n for n in names if n.lower().endswith(video_exts)])


def rescue_missing_cameras(
    base_path: str,
    donor_cam_id: Optional[int] = None,          # if None, first present cam
    n_cams: int = 6,
    clean_target: bool = True,                   # remove target videos before copying
    out_prefix: str = "clonedMissing_",
    video_exts: Tuple[str, ...] = (".mp4", ".avi", ".mov", ".mkv", ".mpg", ".mjpeg", ".mjpg"),
    verify: bool = True,
) -> Dict:
    """
    One call:
      1) detect missing cams;
      2) copy donor videos into each missing Camera{i} folder;
      3) clone params (K,RDistort,TDistort,r,t) + sync.data_sampleID into those cams;
      4) optionally verify videos/params match donor.

    Returns a dict with present/missing, donor, out_mat, and verify results.
    """
    present, missing = detect_missing_cameras(base_path, n_cams=n_cams)
    if not missing:
        return {
            "present": present, "missing": [],
            "donor": donor_cam_id if donor_cam_id in present else (present[0] if present else None),
            "out_mat": _find_label3d_mat(base_path) if present else None,
            "videos_verified": True, "params_verified": True,
            "video_diffs": {}, "param_diffs": {},
        }
    if not present:
        raise RuntimeError("No live cameras detected to use as donor.")

    donor = donor_cam_id if (donor_cam_id in present) else present[0]

    # 1) copy videos
    video_summary = copy_videos_from_donor(
        base_path=base_path,
        donor_cam_id=donor,
        targets=missing,
        n_cams=n_cams,
        video_exts=video_exts,
        clean_target=clean_target,
        sidecars=tuple(),   # add sidecars if you want to copy metadata.csv too
    )

    # 2) clone params/sync into mat
    out_mat = clone_missing_cameras_in_mat(
        base_path=base_path,
        donor_cam_id=donor,
        n_cams=n_cams,
        out_prefix=out_prefix,
        precomputed_missing=missing,   # reuse detection
        fields_to_copy=("K", "RDistort", "TDistort", "r", "t"),
        sync_fields_to_copy=("data_sampleID",),
    )

    # 3) verification (optional)
    videos_ok, params_ok = True, True
    video_diffs: Dict[int, Dict[str, List[str]]] = {}
    param_diffs: Dict[int, List[str]] = {}

    if verify:
        # video filename equality per target
        vids_dir = os.path.join(base_path, "videos")
        donor_dir = os.path.join(vids_dir, f"Camera{donor}")
        donor_files = _list_videos(donor_dir, video_exts)
        donor_set = set(donor_files)

        for cam in missing:
            tgt_dir = os.path.join(vids_dir, f"Camera{cam}")
            tgt_files = _list_videos(tgt_dir, video_exts)
            tgt_set = set(tgt_files)
            if tgt_set != donor_set:
                videos_ok = False
                video_diffs[cam] = {
                    "missing_in_target": sorted(list(donor_set - tgt_set)),
                    "extra_in_target": sorted(list(tgt_set - donor_set)),
                }

        # params + sync equality vs donor (exact array equality)
        m = sio.loadmat(out_mat, struct_as_record=False, squeeze_me=False)
        P = m["params"]
        sync = m.get("sync", None)
        _, name_to_idx = _camname_map(m)
        donor_idx = _camid_to_idx(name_to_idx, donor)
        rec_src = _strip1(_get_rec(P, donor_idx))

        def _eq(a, b):
            a = _strip1(a); b = _strip1(b)
            try:
                return np.array_equal(np.asarray(a), np.asarray(b))
            except Exception:
                return False

        fields = ("K", "RDistort", "TDistort", "r", "t")
        for cam in missing:
            dst_idx = _camid_to_idx(name_to_idx, cam)
            rec_dst = _strip1(_get_rec(P, dst_idx))
            diffs = [f for f in fields if not (_eq(getattr(rec_dst, f), getattr(rec_src, f)) if hasattr(rec_dst, f) and hasattr(rec_src, f) else False)]

            # sync.data_sampleID
            if sync is not None:
                s_src = _strip1(_get_sync_cell(sync, donor_idx))
                s_dst = _strip1(_get_sync_cell(sync, dst_idx))
                if (isinstance(s_src, np.void) and isinstance(s_dst, np.void)
                        and s_src.dtype.names and s_dst.dtype.names
                        and ("data_sampleID" in s_src.dtype.names) and ("data_sampleID" in s_dst.dtype.names)):
                    if not _eq(s_src["data_sampleID"], s_dst["data_sampleID"]):
                        diffs.append("sync.data_sampleID")
                # if sync missing fields, treat as diff
                else:
                    diffs.append("sync.data_sampleID")

            if diffs:
                params_ok = False
                param_diffs[cam] = diffs

    return {
        "present": present,
        "missing": missing,
        "donor": donor,
        "video_copy_summary": video_summary,
        "out_mat": out_mat,
        "videos_verified": videos_ok,
        "params_verified": params_ok,
        "video_diffs": video_diffs,
        "param_diffs": param_diffs,
    }

# Example:
# base = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_07/2social_mini_20241015pmcr2_single_AO_13_24"
# res = rescue_missing_cameras(base, donor_cam_id=None, clean_target=True, verify=True)
# res
